# Golden Test Case Generator for READ-MAS Agents

Generates agent-specific DeepEval goldens from rSDE-Bench, DevBench, and PURE datasets.
Each agent gets goldens matching its input-output structure, split into train/eval for internal agents (60/40) or train/eval/benchmark (20/20/60) for the end-to-end agents.

## Agent Golden Sources
| Agent | Source | Input | Expected Output | Data Splits |
|-------|--------|-------|-----------------|-------------|
| Collector | rSDE-Bench | NL query | `{FRs, NFRs}` | 60/40 |
| Analyzer | rSDE-Bench | CollectorOutputModel | `{useCases, domainClasses, ...}` | 60/40 |
| Specifier | PURE docs | SpecifierInputModel (extracted from same doc) | SRS document | 60/40 |
| Designer | DevBench | SRS text | `{systemArchitecture, fileStructure, componentDesign}` | 20/20/60 |
| Documenter | DevBench | DesignerOutputModel | Design document | 20/20 |
| Single | DevBench | NL query | Design document | 20/20/60 |
| READ (Multi-agent) | DevBench | NL query | Design document | 20/20/60 |

In [14]:
import json
import re
import os
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# Base paths
BASE = Path('../../datasets')
RSDE_WEBSITE = BASE / 'rSDE-Bench' / 'website'
RSDE_GAME = BASE / 'rSDE-Bench' / 'game'
DEVBENCH_PYTHON_PATH = BASE / 'DevBench' / 'benchmark_data' / 'python'
DEVBENCH_JS_PATH = BASE / 'DevBench' / 'benchmark_data' / 'javascript'
DEVBENCH_JAVA_PATH = BASE / 'DevBench' / 'benchmark_data' / 'java'
DEVBENCH_CPP_PATH = BASE / 'DevBench' / 'benchmark_data' / 'cpp'
PURE_DOCS = BASE / 'pure' / 'requirements'
GOLDENS_OUT = Path('../../data/goldens')

# Ensure output dirs exist
for agent in ['collector_agent', 'analyzer_agent', 'specifier_agent',
              'designer_agent', 'documenter_agent', 'read_agent', 'single_agent']:
  (GOLDENS_OUT / agent).mkdir(parents=True, exist_ok=True)

print('Output directories created.')

Output directories created.


---
## 1. Parse rSDE-Bench Specs
Extract structured information from each markdown spec.

In [15]:
def parse_rsde_spec(filepath):
    """Parse an rSDE-Bench markdown spec into structured components.

    Handles all format variants observed across 45 website and 8 game specs:

    Page Design (overview labels):
      - **Overview**: / **Overview:** (most files)
      - **Content & Functionality:** / **Content and Functionality**: (FitnessEquipmentRental, etc.)
      - **Description**: (some files)

    Page Design (element ID formats):
      1. `id` (HTML `<tag>` ...) — CharitableGivingPlatform style
      2. - `id`: description  /  - `id` - description — DigitalArtworkGallery, FitnessEquipmentRental
      3. ID: `id` — EcoFriendlyLivingTips, OnlineCulturalFestivals
      4. **Element ID:** `id`  /  **ID:** `id` — HealthConsultationPlatform
      5. **ID:** id_name (no backticks) — MotivationalQuotesApp

    Data Storage (filename formats):
      1. **File Name**: `filename.ext` / **File Name:** `filename.ext` — CharitableGivingPlatform, EcoFriendlyLivingTips
      2. backtick-quoted `data/filename.ext` — FitnessEquipmentRental
      3. **X Data File (`filename.ext`)** — DigitalArtworkGallery, GourmetFoodSubscription
      4. (filename.ext) / (data/filename.ext) in parens — HealthConsultationPlatform, OnlineCulturalFestivals
      5. **Filename.txt** bold heading — MotivationalQuotesApp
    """
    content = filepath.read_text(encoding='utf-8')
    name = filepath.stem
    spec_type = filepath.parent.name  # 'website' or 'game'

    # Extract objective
    obj_match = re.search(r'## \d+\.\s*Objective\n(.+?)(?=\n## )', content, re.DOTALL)
    objective = obj_match.group(1).strip() if obj_match else ''

    # Extract language
    lang_match = re.search(r'## \d+\.\s*Language\n(.+?)(?=\n## )', content, re.DOTALL)
    language = lang_match.group(1).strip() if lang_match else 'Python'

    # Extract page designs (each ### subsection)
    pages = []
    page_section = re.search(r'## \d+\.\s*Page Design\n(.+?)(?=\n## \d)', content, re.DOTALL)
    if page_section:
        page_blocks = re.split(r'(?=### \d+\.\d+)', page_section.group(1))
        for block in page_blocks:
            title_match = re.search(r'### \d+\.\d+\s+(.+)', block)
            if not title_match:
                continue
            title = title_match.group(1).strip()

            # Extract overview — colon may appear inside or outside the bold markers
            overview = ''
            for label_pat in [
                r'\*\*Overview:?\*\*:?\s*([^\n]+)',
                r'\*\*Page Overview:?\*\*:?\s*([^\n]+)',
                r'\*\*Content\s*(?:&|and)\s*Functionality:?\*\*:?\s*([^\n]+)',
                r'\*\*Description:?\*\*:?\s*([^\n]+)',
            ]:
                m = re.search(label_pat, block, re.IGNORECASE)
                if m:
                    candidate = m.group(1).strip()
                    # Skip empty or lines that are just nested list starters
                    if candidate and not candidate.startswith(('-', '*', '•')):
                        overview = candidate
                        break

            # Extract element IDs — multiple patterns covering all observed spec formats
            elements = []
            seen_ids: set = set()

            def add_element(id_val, tag=''):
                if id_val not in seen_ids:
                    seen_ids.add(id_val)
                    elements.append((id_val, tag))

            # Pattern 1: `id` (HTML `<tag>` ...) — CharitableGivingPlatform
            for m in re.finditer(r'`([\w-]+)`\s*\(HTML\s*`<(\w+)>`[^)]*\)', block):
                add_element(m.group(1), m.group(2))

            # Pattern 2: - `id`: description  or  - `id` - description  (list items)
            # Covers DigitalArtworkGallery, FitnessEquipmentRental, OnlineCulturalFestivals
            for m in re.finditer(r'^\s*[-*]\s+`([\w-]+)`\s*[-:]', block, re.MULTILINE):
                add_element(m.group(1))

            # Pattern 3: ID: `id`  — EcoFriendlyLivingTips, OnlineCulturalFestivals
            for m in re.finditer(r'\bID:\s+`([\w-]+)`', block):
                add_element(m.group(1))

            # Pattern 4: **Element ID:** `id`  or  **ID:** `id`  — HealthConsultationPlatform
            for m in re.finditer(r'\*\*(?:Element\s+)?ID:?\*\*:?\s*`([\w-]+)`', block, re.IGNORECASE):
                add_element(m.group(1))

            # Pattern 5: **ID:** id_name  (no backticks) — MotivationalQuotesApp
            for m in re.finditer(r'\*\*ID:?\*\*:?\s+([\w-]+)', block, re.IGNORECASE):
                id_val = m.group(1)
                # Accept only identifier-style values (has underscore or starts lowercase)
                if '_' in id_val or (id_val[0].islower() and len(id_val) > 2):
                    add_element(id_val)

            pages.append({'title': title, 'overview': overview, 'elements': elements})

    # Extract data files from the Data Storage section
    data_files = []
    data_section = re.search(r'## \d+\.\s*Data Storage\n(.+?)(?=\n## \d|\Z)', content, re.DOTALL)
    if data_section:
        data_text = data_section.group(1)
        seen_filenames: set = set()
        candidates = []

        # Pattern 1: backtick-quoted `filename.ext` or `path/filename.ext` — strip path prefix
        for m in re.finditer(r'`([^`\s]+\.(?:txt|json|csv|log))`', data_text):
            fname = m.group(1).rsplit('/', 1)[-1]
            candidates.append((fname, m.start()))

        # Pattern 2: (filename.ext) or (data/filename.ext) in parentheses
        # Covers HealthConsultationPlatform, OnlineCulturalFestivals
        for m in re.finditer(r'\((?:[^()\s]*/)?([^()\s/]+\.(?:txt|json|csv|log))\)', data_text):
            candidates.append((m.group(1), m.start()))

        # Pattern 3: **Filename.txt** as a bold section heading — MotivationalQuotesApp
        for m in re.finditer(r'\*\*([A-Za-z_][\w]*\.(?:txt|json|csv|log))\*\*:?', data_text):
            candidates.append((m.group(1), m.start()))

        # Sort by position; deduplicate case-insensitively
        candidates.sort(key=lambda x: x[1])
        for filename, pos in candidates:
            key = filename.lower()
            if key in seen_filenames:
                continue
            seen_filenames.add(key)

            # Find format description in the ~600 chars after this mention
            after = data_text[pos:pos + 600]
            fmt = ''
            fields = []

            fmt_m = (
                # **Data Format**: `fields`  or  **Data Format:** `fields`
                re.search(r'\*{0,2}Data Format:?\*{0,2}:?\s*`([^`]+)`', after) or
                # **Content Format**: followed by a code block (MotivationalQuotesApp)
                re.search(r'\*{0,2}Content Format:?\*{0,2}:?\s*\n\s*```\s*\n([^\n]+)', after) or
                # Format: `fields`
                re.search(r'\bFormat:\s*`([^`]+)`', after) or
                # Format: fields  (unquoted, to end of line, stops before list markers)
                re.search(r'\bFormat:\s+([^\n`*\\]+)', after)
            )
            if fmt_m:
                fmt = fmt_m.group(1).strip()
                fields = [f.strip() for f in re.split(r'[,|]', fmt) if f.strip()]
            else:
                # Fallback: first line of nearest code block — use only if it looks like a header
                code_m = re.search(r'```\s*\n([^\n]+)', after)
                if code_m:
                    first_line = code_m.group(1).strip()
                    parts = [p.strip() for p in re.split(r'[,|]', first_line) if p.strip()]
                    # Accept as header only if all parts are pure identifier-style names (no digits)
                    if (len(parts) > 1 and
                            all(re.match(r'^[A-Za-z_][A-Za-z0-9_]*$', p) for p in parts)):
                        fields = parts
                        fmt = first_line

            data_files.append({'filename': filename, 'format': fmt, 'fields': fields})

    elif spec_type == 'game':
        # Game specs embed log file specs in requirements text
        log_match = re.search(r'"([\w.]+\.log)"', content)
        if log_match:
            data_files.append({
                'filename': log_match.group(1),
                'format': 'JSON event log',
                'fields': ['timestamp', 'EVENT_TYPE', 'game_state'],
            })

    return {
        'name': name,
        'type': spec_type,
        'objective': objective,
        'language': language,
        'pages': pages,
        'data_files': data_files,
        'full_content': content,
    }


# Parse all specs
rsde_specs = []
for f in sorted(RSDE_WEBSITE.glob('*.md')):
    rsde_specs.append(parse_rsde_spec(f))
for f in sorted(RSDE_GAME.glob('*.md')):
    rsde_specs.append(parse_rsde_spec(f))

print(f'Parsed {len(rsde_specs)} rSDE-Bench specs')
print(f'  Website: {sum(1 for s in rsde_specs if s["type"] == "website")}')
print(f'  Game:    {sum(1 for s in rsde_specs if s["type"] == "game")}')

# Verify page and data_file extraction across all website specs
website_specs = [s for s in rsde_specs if s['type'] == 'website']
total_pages = sum(len(s['pages']) for s in website_specs)
total_elements = sum(len(p['elements']) for s in website_specs for p in s['pages'])
total_data_files = sum(len(s['data_files']) for s in website_specs)
missing_data_files = [s['name'] for s in website_specs if not s['data_files']]
print(f'\nWebsite extraction summary:')
print(f'  Total pages:      {total_pages}')
print(f'  Total elements:   {total_elements}')
print(f'  Total data_files: {total_data_files}')
if missing_data_files:
    print(f'  Specs with 0 data_files: {missing_data_files}')

# Show a representative sample
sample = rsde_specs[0]
print(f'\nSample: {sample["name"]}')
print(f'  Pages ({len(sample["pages"])}): {[p["title"] for p in sample["pages"]]}')
print(f'  data_files ({len(sample["data_files"])}):')
for df in sample['data_files']:
    print(f'    {df["filename"]}: fields={df["fields"]}')

Parsed 53 rSDE-Bench specs
  Website: 45
  Game:    8

Website extraction summary:
  Total pages:      242
  Total elements:   702
  Total data_files: 135
  Specs with 0 data_files: ['TaskManager']

Sample: CharitableGivingPlatform
  Pages (3): ['Login Page', 'Dashboard Page', 'Charity Details Page']
  data_files (3):
    users.txt: fields=[]
    contributions.txt: fields=[]
    charities.txt: fields=[]


---
## 2. Parse DevBench Projects

In [16]:
def parse_devbench_project(project_dir):
    """Parse a DevBench project's artifacts."""
    result = {'name': project_dir.name}

    for doc_name in ['PRD.md', 'UML_class.md', 'UML_sequence.md', 'architecture_design.md']:
        path = project_dir / doc_name
        language = project_dir.parent.name
        if path.exists():
            result[doc_name.replace('.md', '')] = path.read_text(encoding='utf-8')
        else:
            result[doc_name.replace('.md', '')] = ''
        
        result['language'] = language

    return result


devbench_paths = sorted([d for p in [DEVBENCH_PYTHON_PATH, DEVBENCH_JS_PATH, DEVBENCH_JAVA_PATH, DEVBENCH_CPP_PATH] for d in p.iterdir() if d.is_dir()])

devbench_projects = []
for d in devbench_paths:
    devbench_projects.append(parse_devbench_project(d))

print(f'Parsed {len(devbench_projects)} DevBench projects')
for p in devbench_projects:
    print(f'  {p["name"]}: PRD={len(p["PRD"])} chars, UML_class={len(p["UML_class"])} chars, Language: {p["language"]}')

Parsed 22 DevBench projects
  area_calculation: PRD=3314 chars, UML_class=529 chars, Language: cpp
  graph-cpp: PRD=5834 chars, UML_class=1773 chars, Language: cpp
  logistic_system: PRD=11495 chars, UML_class=1352 chars, Language: cpp
  people_management: PRD=5781 chars, UML_class=1696 chars, Language: cpp
  xlsx2csv: PRD=4500 chars, UML_class=2251 chars, Language: cpp
  Actor_relationship_game: PRD=5888 chars, UML_class=1822 chars, Language: java
  idcenter: PRD=3803 chars, UML_class=1240 chars, Language: java
  image-similarity: PRD=6375 chars, UML_class=1351 chars, Language: java
  java_heap: PRD=5465 chars, UML_class=2089 chars, Language: java
  redis-cache: PRD=5676 chars, UML_class=3282 chars, Language: java
  listen-now-frontend: PRD=7053 chars, UML_class=358 chars, Language: javascript
  login-registration: PRD=5237 chars, UML_class=813 chars, Language: javascript
  ArXiv_digest: PRD=4637 chars, UML_class=1399 chars, Language: python
  Hybrid_Images: PRD=2625 chars, UML_class=

---
## 3. Data Split
Split rSDE-Bench into train/eval and DevBench into train/eval/benchmark sets.

In [ ]:
# Split rSDE-Bench website and game specs 
website_specs = [s for s in rsde_specs if s['type'] == 'website']
game_specs = [s for s in rsde_specs if s['type'] == 'game']

# Train/eval split by 60/40 ratio for both website and game specs.
rsde_splits = {
    'train': website_specs[:27] + game_specs[:6],
    'eval': website_specs[27:] + game_specs[6:]
}

# Split DevBench
# Stratify on the project language field
# Use train/eval/benchmark split by 20/20/60 ratio
proj_lang_array = [p['language'] for p in devbench_projects]
train, tmp = train_test_split(devbench_projects, test_size=0.8, stratify=proj_lang_array, random_state=42)
tmp_lang_array = [p['language'] for p in tmp]
eval, benchmark = train_test_split(tmp, test_size=0.75, stratify=tmp_lang_array, random_state=42)
devbench_splits = {
    'train': train,
    'eval': eval,
    'benchmark': benchmark,
}

print('=== Data Split ===')
for split_name in ['train', 'eval', 'benchmark']:
    rsde_names = [s['name'] for s in rsde_splits[split_name]]  if split_name != 'benchmark' else []
    dev_names = [p['name'] for p in devbench_splits[split_name]]
    print(f'\n{split_name}:')
    print(f'  rSDE-Bench ({len(rsde_names)}): {rsde_names}')
    print(f'  DevBench ({len(dev_names)}): {dev_names}')

=== Data Split ===

train:
  rSDE-Bench (33): ['CharitableGivingPlatform', 'DailyHealthTips', 'DailyJournalApp', 'DigitalArtworkGallery', 'DigitalStorytellingPlatform', 'EcoFriendlyLivingTips', 'ElderCareResources', 'EventPlanner', 'ExpenseTracker', 'FitnessChallenges', 'FitnessEquipmentRental', 'FitnessTracker', 'FreelancerMarketplace', 'GardeningForBeginners', 'GourmetFoodSubscription', 'GreenLivingGuide', 'HealthConsultationPlatform', 'MotivationalQuotesApp', 'MovieRecommendationSystem', 'MusicCollaborator', 'MusicFestivalDirectory', 'NoteTakingApp', 'NutritionInformationHub', 'OnlineCulturalExchange', 'OnlineCulturalFestivals', 'OnlineLibraryManagementSystem', 'OnlineShoppingCenter', 'battle_of_balls_game', 'bomerman_game', 'brick_breaker_game', 'ghostly_game', 'racing_game', 'sokoban_game']
  DevBench (4): ['hone', 'ArXiv_digest', 'graph-cpp', 'java_heap']

eval:
  rSDE-Bench (20): ['OnlineTherapeuticJournaling', 'OnlineThriftStore', 'OnlineVintageMarket', 'ParentingAdviceForum', 

---
## 4. Generate Collector Agent Goldens
Input: Natural language query derived from objective. Expected output: `{FRs: [...], NFRs: [...]}`

In [18]:
def _generate_game_collector_golden(spec):
    """Generate a collector golden from a game spec (Task:/Requirements: format).

    FRs: gameplay mechanics (controls, collision, win/lose conditions, game objects).
    NFRs: log file requirements, GUI constraint, language.
    """
    content = spec['full_content']

    # Extract task description as the NL query
    task_match = re.search(r'^Task:\s*(.+)$', content, re.MULTILINE)
    query = (
        task_match.group(1).strip()
        if task_match
        else f"Design a {spec['name'].replace('_', ' ')} application."
    )

    # Isolate the requirements block
    req_section = re.search(r'Requirements:\n(.*)', content, re.DOTALL)
    frs, nfrs = [], []
    if req_section:
        req_text = req_section.group(1).strip()
        # Split on lines that begin a new numbered item ("1. ", "2. ", …)
        req_blocks = re.split(r'\n(?=\d+\.\s)', req_text)
        log_keywords = ('game.log', 'log file', 'log entry', 'log format')
        for block in req_blocks:
            block = block.strip()
            if not block:
                continue
            # Strip leading "N. " prefix
            text = re.sub(r'^\d+\.\s*', '', block).strip()
            if any(kw in text.lower() for kw in log_keywords):
                nfrs.append(text)
            else:
                frs.append(text)

    # Prepend language and GUI NFRs
    prefix_nfrs = []
    if spec['language']:
        prefix_nfrs.append(f"The game will be developed using the {spec['language']} programming language.")
    if 'gui' in query.lower():
        prefix_nfrs.append("The application shall have a graphical user interface (GUI).")
    nfrs = prefix_nfrs + nfrs

    return {
        'input': f"{query}\n\n{spec['full_content']}",
        'actual_output': None,
        'expected_output': json.dumps({'FRs': frs, 'NFRs': nfrs}),
        'context': [spec['full_content']],
        'source_file': spec['name'],
    }


def generate_collector_golden(spec):
    """Generate a collector agent golden from an rSDE-Bench spec."""
    if spec['type'] == 'game':
        return _generate_game_collector_golden(spec)

    # --- Website specs ---
    query = spec['objective']
    if not query:
        query = f"Design a {spec['name'].replace('_', ' ')} application."

    frs = []
    nfrs = []

    # FRs from page designs (each page = functional capability)
    for page in spec['pages']:
        if page['overview']:
            frs.append(f"The system shall provide a {page['title']}: {page['overview']}")
        for elem_id, elem_tag in page['elements']:
            frs.append(f"The {page['title']} shall include a {elem_tag} element ({elem_id}).")

    # NFRs: language constraint, storage, platform
    if spec['language']:
        nfrs.append(f"{spec['language']}")
    nfrs.append("The system shall store data locally using text files.")
    nfrs.append("The application shall allow access only to authenticated users.")
    nfrs.append("The page load shall take less than 2 seconds for 95% of authenticated requests.")

    return {
        'input': f"{query}\n\n{spec['full_content']}",
        'actual_output': None,
        'expected_output': json.dumps({'FRs': frs, 'NFRs': nfrs}),
        'context': [spec['full_content']],
        'source_file': spec['name'],
    }


# Generate for each split
for split_name, specs in rsde_splits.items():
    goldens = [generate_collector_golden(s) for s in specs]
    out_path = GOLDENS_OUT / 'collector_agent' / f'{split_name}.json'
    with open(out_path, 'w') as f:
        json.dump(goldens, f, indent=2)
    print(f'Collector {split_name}: {len(goldens)} goldens -> {out_path}')

# Spot-check a game golden
game_sample = next(s for s in rsde_specs if s['type'] == 'game')
g = generate_collector_golden(game_sample)
out = json.loads(g['expected_output'])
print(f"\nGame sample: {game_sample['name']}")
print(f"  Query:  {g['input'][:80]}")
print(f"  FRs ({len(out['FRs'])}):")
for fr in out['FRs']:
    print(f"    - {fr[:80]}")
print(f"  NFRs ({len(out['NFRs'])}):")
for nfr in out['NFRs']:
    print(f"    - {nfr[:80]}")

Collector train: 33 goldens -> ../../data/goldens/collector_agent/train.json
Collector eval: 20 goldens -> ../../data/goldens/collector_agent/eval.json

Game sample: battle_of_balls_game
  Query:  Develop a Battle of Balls Game
  FRs (6):
    - Different balls will be distinguished by their colors.
    - The player moves using the up, down, left, and right arrow keys, but with the pl
    - When two balls collide, the smaller ball (with a smaller radius) will be consume
    - The game ends if the player's ball is consumed.
    - In addition to the player's ball, initialize four enemy balls that have the same
    - Small, non-player, and enemy balls will continuously spawn on the map (much smal
  NFRs (2):
    - The game will be developed using the Python programming language.
    - When the game starts, a new log file named "game.log" should be created to recor


---
## 5. Generate Analyzer Agent Goldens
Input: CollectorOutputModel JSON. Expected output: AnalyzerOutputModel JSON.

In [ ]:

def _overview_to_use_case(overview, page_title):
  """Convert a page overview string to an action-oriented use case."""
  m = re.search(
      r'(?:allows?|enables?|permits?)\s+(?:users?|the user)\s+to\s+(.+?)(?:\.|,|$)',
      overview, re.IGNORECASE,
  )
  if m:
    return f"User {m.group(1).strip()}"
  m = re.search(
      r'(?:provides?)\s+(?:users?|the user)\s+with\s+(.+?)(?:\.|,|$)',
      overview, re.IGNORECASE,
  )
  if m:
    return f"User accesses {m.group(1).strip()}"
  title = page_title.replace(' Page', '')
  return f"User navigates to {title}: {overview[:100]}"


def _to_class_name(filename):
  """Convert a data filename to a CamelCase class name (e.g. users.txt → Users)."""
  stem = filename.rsplit('.', 1)[0]
  return ''.join(w.capitalize() for w in re.split(r'[_\-\s]', stem))


def _to_entity_name(filename):
  """Convert a data filename to UPPER_SNAKE for ER diagrams (e.g. users.txt → USERS)."""
  stem = filename.rsplit('.', 1)[0].upper()
  return re.sub(r'[\-\s]', '_', stem)


def _detect_game_entities(content):
  """Return (class_name, [attr, ...]) pairs for game-specific domain entities."""
  c = content.lower()
  if 'ball' in c:
    return [
        ('Ball',       ['+float[] position', '+float radius', '+String color']),
        ('PlayerBall', ['+bool isConsumed']),
        ('EnemyBall',  ['+bool isFixed', '+bool isActive']),
    ]
  if 'mario' in c or ('brick' in c and 'breaker' in c):
    return [
        ('Player', ['+float[] position', '+int lives', '+int score']),
        ('Brick',  ['+int[] position', '+bool isDestroyed']),
        ('Ball',   ['+float[] position', '+float[] velocity']),
    ]
  if 'sokoban' in c or ('box' in c and 'push' in c):
    return [
        ('Player', ['+int[] position']),
        ('Box',    ['+int[] position', '+bool onTarget']),
        ('Target', ['+int[] position']),
    ]
  if 'tank' in c:
    return [
        ('Tank',   ['+float[] position', '+float health', '+String color']),
        ('Bullet', ['+float[] position', '+float[] velocity']),
    ]
  if 'bomber' in c or 'bomberman' in c:
    return [
        ('Player', ['+int[] position', '+int lives']),
        ('Bomb',   ['+int[] position', '+float timer']),
        ('Enemy',  ['+int[] position', '+bool isAlive']),
    ]
  if 'race' in c or 'racing' in c:
    return [
        ('Car',   ['+float[] position', '+float speed', '+String color']),
        ('Track', ['+int laps', '+float length']),
    ]
  if 'ghost' in c or 'pacman' in c:
    return [
        ('Player', ['+float[] position', '+int score', '+int lives']),
        ('Ghost',  ['+float[] position', '+bool isVulnerable']),
    ]
  return [
      ('Player',     ['+float[] position', '+int score', '+bool isAlive']),
      ('GameEntity', ['+float[] position']),
  ]


# website golden

def _generate_website_analyzer_golden(spec):
  """Generate an analyzer golden from a website rSDE-Bench spec."""
  collector_golden = generate_collector_golden(spec)
  collector_output = json.loads(collector_golden['expected_output'])
  input_data = json.dumps(collector_output)
  frs  = collector_output['FRs']
  nfrs = collector_output['NFRs']

  # 1. Use cases — one per page overview, plus each non-trivial button action
  use_cases = []
  for page in spec['pages']:
    if page['overview']:
      use_cases.append(_overview_to_use_case(page['overview'], page['title']))
    for elem_id, elem_tag in page['elements']:
      if elem_tag != 'button':
        continue
      lower_id = elem_id.lower()
      if any(skip in lower_id for skip in ('back', 'return', 'cancel', 'close', 'navigate')):
        continue
      readable = (
          re.sub(r'([A-Z])', r' \1', elem_id)
          .strip().lower()
          .replace('button', '').strip()
      )
      if readable:
        use_cases.append(
            f"User {readable} on {page['title'].replace(' Page', '')} page"
        )

  # shared-field lookup (used for both class diagram and ER relationships)
  field_to_files: dict = {}
  for df in spec['data_files']:
    for field in df['fields']:
      field_to_files.setdefault(field, []).append(df['filename'])

  # 2. Domain classes — derived from data files with their parsed fields
  domain_classes = 'classDiagram\n'
  for df in spec['data_files']:
    cls = _to_class_name(df['filename'])
    domain_classes += f'  class {cls} {{\n'
    for field in df['fields']:
      ftype = (
          'float' if re.search(r'(amount|price|count|num|size|age)$', field, re.I)
          else 'String'
      )
      domain_classes += f'    +{ftype} {field}\n'
    if not df['fields']:
      domain_classes += '    +String id\n'
    domain_classes += '  }\n'
  # associations via shared fields
  added_cls_rels: set = set()
  for field, files in field_to_files.items():
    if len(files) >= 2:
      ca, cb = _to_class_name(files[0]), _to_class_name(files[1])
      key = frozenset([ca, cb])
      if key not in added_cls_rels:
        added_cls_rels.add(key)
        domain_classes += f'  {ca} "1" --> "*" {cb} : "{field}"\n'

  # 3. Business rules — auth constraints + per-file format rules
  business_rules = []
  if any(p['title'].lower().startswith('login') for p in spec['pages']):
    business_rules.append('Only authenticated users may access protected pages.')
    business_rules.append(
        'Users must supply valid credentials (username and password) to log in.'
    )
  business_rules.append(
      'All application data must be persisted in local text files within the data/ directory.'
  )
  for df in spec['data_files']:
    if df['fields']:
      business_rules.append(
          f"Records in '{df['filename']}' must conform to the format: {', '.join(df['fields'])}."
      )

  # 4. Data model — ER diagram with field-level detail
  data_model = 'erDiagram\n'
  entity_names: dict = {}
  for df in spec['data_files']:
    entity = _to_entity_name(df['filename'])
    entity_names[df['filename']] = entity
    data_model += f'  {entity} {{\n'
    for field in df['fields']:
      ftype = (
          'float' if re.search(r'(amount|price|count|num|size|age)$', field, re.I)
          else 'string'
      )
      data_model += f'    {ftype} {field}\n'
    if not df['fields']:
      data_model += '    string id\n'
    data_model += '  }\n'
  added_er: set = set()
  for field, files in field_to_files.items():
    if len(files) >= 2:
      ea = entity_names.get(files[0])
      eb = entity_names.get(files[1])
      if ea and eb:
        key = frozenset([ea, eb])
        if key not in added_er:
          added_er.add(key)
          data_model += f'  {ea} ||--o{{ {eb} : "has"\n'

  # 5. Traceability — map every FR to the nearest use case
  n_uc = max(len(use_cases), 1)
  traceability = [f'FR-{i+1} -> UC-{(i % n_uc) + 1}' for i in range(len(frs))]

  # 6. Validation
  validation = [
      'All functional requirements are traceable to at least one use case.',
      'All domain classes are derived from data storage specifications.',
      f'Validated {len(frs)} FRs and {len(nfrs)} NFRs with no redundancies detected.',
  ]

  return {
      'input': input_data,
      'actual_output': None,
      'expected_output': json.dumps({
          'useCases':      use_cases,
          'domainClasses': domain_classes,
          'businessRules': business_rules,
          'dataModel':     data_model,
          'traceability':  traceability,
          'validation':    validation,
      }),
      'context':     [spec['full_content'][:2000]],
      'source_file': spec['name'],
  }


# game golden

def _generate_game_analyzer_golden(spec):
  """Generate an analyzer golden from a game rSDE-Bench spec."""
  collector_golden = generate_collector_golden(spec)
  collector_output = json.loads(collector_golden['expected_output'])
  input_data = json.dumps(collector_output)
  frs     = collector_output['FRs']
  content = spec['full_content']

  # 1. Use cases — player-action FRs become "Player: …"; system events become "System: …"
  player_kw = ('player', 'arrow key', 'move', 'control', 'user', 'press')
  system_kw = ('game end', 'game ends', 'game over', 'game start', 'game starts',
               'initialize', 'spawn', 'initial state')
  use_cases = []
  for fr in frs:
    fl = fr.lower()
    if any(kw in fl for kw in player_kw):
      use_cases.append(f"Player: {fr[:120]}")
    elif any(kw in fl for kw in system_kw):
      use_cases.append(f"System: {fr[:120]}")
  if not use_cases:
    use_cases = [f"Player interacts with the {spec['name'].replace('_', ' ')} game."]

  # 2. Domain classes — game-type entities + optional logger/state
  entities = _detect_game_entities(content)
  if spec['data_files']:
    entities += [
        ('GameLogger', ['+String logFile', '+String eventType']),
        ('GameState',  ['+String timestamp', '+Object playerState', '+List enemyStates']),
    ]
  domain_classes = 'classDiagram\n'
  for cls_name, attrs in entities:
    domain_classes += f'  class {cls_name} {{\n'
    for attr in attrs:
      domain_classes += f'    {attr}\n'
    domain_classes += '  }\n'
  # inheritance for ball-type games
  cls_names = [e[0] for e in entities]
  if 'Ball' in cls_names and 'PlayerBall' in cls_names:
    domain_classes += '  Ball <|-- PlayerBall\n'
    domain_classes += '  Ball <|-- EnemyBall\n'
  if 'GameLogger' in cls_names:
    domain_classes += '  GameLogger --> GameState : "records"\n'

  # 3. Business rules — categorised from requirement text
  rule_patterns = [
      (r'consume|smaller.*larger|larger.*smaller', 'Collision rule'),
      (r'game end|game over|player.*consumed',     'Game-over condition'),
      (r'log|game\.log|record',                    'Logging requirement'),
      (r'spawn',                                   'Spawn rule'),
      (r'center|reference frame',                  'Positioning rule'),
      (r'arrow key|move.*key|key.*move',           'Control rule'),
      (r'radius',                                  'Size rule'),
  ]
  business_rules = []
  for fr in frs:
    fl = fr.lower()
    for pattern, label in rule_patterns:
      if re.search(pattern, fl):
        business_rules.append(f"{label}: {fr[:120]}")
        break
  if spec['language']:
    business_rules.append(
        f"Language constraint: The game must be implemented in {spec['language']}."
    )

  # 4. Data model — log-file JSON structure as an ER diagram
  data_model = 'erDiagram\n'
  if spec['data_files']:
    data_model += '  GAME_LOG {\n'
    data_model += '    string timestamp\n'
    data_model += '    string EVENT_TYPE\n'
    data_model += '  }\n'
    data_model += '  GAME_STATE {\n'
    data_model += '    float player_x\n'
    data_model += '    float player_y\n'
    data_model += '    float player_radius\n'
    data_model += '  }\n'
    data_model += '  ENEMY {\n'
    data_model += '    float position_x\n'
    data_model += '    float position_y\n'
    data_model += '    float radius\n'
    data_model += '  }\n'
    data_model += '  GAME_LOG ||--|| GAME_STATE : "captures"\n'
    data_model += '  GAME_STATE ||--o{ ENEMY : "contains"\n'
  else:
    data_model += '  GAME_STATE {\n    string status\n    string player\n  }\n'

  # 5. Traceability
  n_uc = max(len(use_cases), 1)
  traceability = [f'FR-{i+1} -> UC-{(i % n_uc) + 1}' for i in range(len(frs))]

  # 6. Validation
  validation = [
      'All functional requirements are traceable to at least one use case.',
      f'Validated {len(frs)} game requirements against domain model entities.',
      'Log file structure is consistent with game event types defined in requirements.',
  ]

  return {
      'input': input_data,
      'actual_output': None,
      'expected_output': json.dumps({
          'useCases':      use_cases,
          'domainClasses': domain_classes,
          'businessRules': business_rules,
          'dataModel':     data_model,
          'traceability':  traceability,
          'validation':    validation,
      }),
      'context':     [spec['full_content'][:2000]],
      'source_file': spec['name'],
  }


# Generate the analyzer agent goldens
def generate_analyzer_golden(spec):
  """Generate an analyzer agent golden from an rSDE-Bench spec."""
  if spec['type'] == 'game':
    return _generate_game_analyzer_golden(spec)
  return _generate_website_analyzer_golden(spec)


# generate and save the goldens

for split_name, specs in rsde_splits.items():
  goldens = [generate_analyzer_golden(s) for s in specs]
  out_path = GOLDENS_OUT / 'analyzer_agent' / f'{split_name}.json'
  with open(out_path, 'w') as f:
    json.dump(goldens, f, indent=2)
  print(f'Analyzer {split_name}: {len(goldens)} goldens -> {out_path}')

# Golden validations

print('\n--- Website spot-check (CharitableGivingPlatform) ---')
website_sample = next(s for s in rsde_specs if s['name'] == 'CharitableGivingPlatform')
wg = generate_analyzer_golden(website_sample)
wo = json.loads(wg['expected_output'])
print(f"  useCases ({len(wo['useCases'])}):")
for uc in wo['useCases']:
  print(f"    - {uc[:90]}")
print(f"  domainClasses snippet:\n{wo['domainClasses'][:300]}")
print(f"  businessRules ({len(wo['businessRules'])}):")
for br in wo['businessRules']:
  print(f"    - {br[:90]}")
print(f"  dataModel snippet:\n{wo['dataModel'][:300]}")

print('\n--- Game spot-check (battle_of_balls_game) ---')
game_sample = next(s for s in rsde_specs if s['name'] == 'battle_of_balls_game')
gg = generate_analyzer_golden(game_sample)
go = json.loads(gg['expected_output'])
print(f"  useCases ({len(go['useCases'])}):")
for uc in go['useCases']:
  print(f"    - {uc[:90]}")
print(f"  domainClasses snippet:\n{go['domainClasses'][:300]}")
print(f"  businessRules ({len(go['businessRules'])}):")
for br in go['businessRules']:
  print(f"    - {br[:90]}")
print(f"  dataModel:\n{go['dataModel']}")


Analyzer train: 33 goldens -> ../../data/goldens/analyzer_agent/train.json
Analyzer eval: 20 goldens -> ../../data/goldens/analyzer_agent/eval.json

--- Website spot-check (CharitableGivingPlatform) ---
  useCases (7):
    - User log in to their accounts to access platform features
    - User login on Login page
    - User accesses an overview of available charities
    - User charity details on Dashboard page
    - User logout on Dashboard page
    - User navigates to Charity Details: This page provides detailed information about a selecte
    - User donate on Charity Details page
  domainClasses snippet:
classDiagram
  class Users {
    +String id
  }
  class Contributions {
    +String id
  }
  class Charities {
    +String id
  }

  businessRules (3):
    - Only authenticated users may access protected pages.
    - Users must supply valid credentials (username and password) to log in.
    - All application data must be persisted in local text files within the data/ directory.
  dat

---
## 6. Generate Specifier Agent Goldens
Input: SpecifierInputModel JSON with FRs, NFRs, and use cases extracted from a PURE SRS document.
Expected output: The same PURE SRS document.

Source: PURE holdout docs — structured requirements are parsed from each doc and used as input,
while the same doc's full text serves as the expected SRS output. This ensures input and output
are always about the same system.

In [20]:
import pdfplumber
from docx import Document

# PURE holdout docs (from constants.py)
PURE_HOLDOUT = {
    "2000 - nasa x38", "2001 - beyond", "2001 - ctc network",
    "2001 - elsfork", "2001 - libra", "2001 - npac",
    "2001 - space fractions", "2002 - evla back", "2002 - evla corr",
    "2003 - agentmom", "2003 - pnnl", "2004 - grid bgc",
    "2004 - rlcs", "2004 - watcom", "2005 - pontis",
    "2005 - clarus low", "2006 - stewards", "2007 - puget sound",
    "2007 - water use", "2008 - viper", "2008 - vub",
    "2009 - email", "2009 - gaia", "2009 - inventory 2.0",
    "2009 - model manager", "2009 - warc III", "2010 - fishing",
    "2010 - gparted", "2010 - home 1.3", "2010 - mashboot",
}


def extract_pure_doc(filepath):
    """Extract text from a PURE document."""
    ext = filepath.suffix.lower()
    if ext == '.pdf':
        text_parts = []
        try:
            with pdfplumber.open(filepath) as pdf:
                for page in pdf.pages:
                    t = page.extract_text()
                    if t:
                        text_parts.append(t)
        except Exception as e:
            print(f'Error reading {filepath.name}: {e}')
        return '\n\n'.join(text_parts)
    elif ext == '.docx':
        try:
            doc = Document(str(filepath))
            return '\n\n'.join(p.text for p in doc.paragraphs if p.text.strip())
        except Exception as e:
            print(f'Error reading {filepath.name}: {e}')
            return ''
    return ''


# Load holdout PURE docs
pure_holdout_docs = []
for filepath in sorted(PURE_DOCS.iterdir()):
    base = filepath.stem
    if base in PURE_HOLDOUT:
        text = extract_pure_doc(filepath)
        if len(text) > 200:
            pure_holdout_docs.append({'name': base, 'text': text, 'path': str(filepath)})

print(f'Loaded {len(pure_holdout_docs)} PURE holdout documents')
for doc in pure_holdout_docs[:5]:
    print(f'  {doc["name"]}: {len(doc["text"])} chars')
    
print(f'Average length of document: {sum([len(doc["text"]) for doc in pure_holdout_docs]) / len(pure_holdout_docs)}')

# Split PURE holdout docs into train/eval (80/20)
n_pure = len(pure_holdout_docs)
n_train = int(n_pure * 0.6)
pure_splits = {
    'train': pure_holdout_docs[:n_train],
    'eval': pure_holdout_docs[n_train:],
}
print(f'\nPURE splits: train={len(pure_splits["train"])}, eval={len(pure_splits["eval"])}')

Loaded 30 PURE holdout documents
  2000 - nasa x38: 70664 chars
  2001 - beyond: 76701 chars
  2001 - ctc network: 44205 chars
  2001 - elsfork: 119148 chars
  2001 - libra: 29410 chars
Average length of document: 77704.8

PURE splits: train=18, eval=12


In [21]:
def parse_pure_doc_requirements(text, doc_name):
    """Extract FRs, NFRs, and use cases from a PURE SRS document's own text.

    Handles the variety of heading styles and numbering schemes found across
    the PURE corpus (IEEE 830-style, custom section numbering, plain prose).
    """
    text_norm = re.sub(r'\r\n', '\n', text)
    frs, nfrs, use_cases = [], [], []

    # ── Functional Requirements ───────────────────────────────────────────────
    fr_section_match = re.search(
        r'(?:^|\n)[ \t]*(?:\d[\d.]*\s+)?'
        r'(?:Functional\s+Requirements?|System\s+Requirements?|Software\s+Requirements?)'
        r'[ \t]*\n(.*?)'
        r'(?=\n[ \t]*(?:\d[\d.]*\s+)?(?:Non.?[Ff]unctional|Performance|Constraint|Quality'
        r'|External\s+Interface|Design\s+Constraint|Safety|Security|Interface|\Z))',
        text_norm, re.DOTALL | re.IGNORECASE,
    )
    fr_text = fr_section_match.group(1) if fr_section_match else text_norm[:8000]

    # Numbered items: "3.1.1 The system shall …" or "FR-12: …"
    for m in re.finditer(
        r'(?:^|\n)[ \t]*(?:\d+(?:\.\d+)+[ \t]+|[A-Z]{2,5}-\d+[: \t]+)(.{20,300})',
        fr_text, re.MULTILINE,
    ):
        req = m.group(1).strip().rstrip('.')
        skip_words = ('introduction', 'overview', 'purpose', 'scope', 'definition',
                      'reference', 'figure', 'table', 'section', 'appendix')
        if len(req) >= 20 and not any(req.lower().startswith(w) for w in skip_words):
            frs.append(req)

    # Fallback: sentences containing "shall" if numbered items are sparse
    if len(frs) < 3:
        for m in re.finditer(
            r'(?:^|\n)[ \t]*(?:[-*•]\s+)?([^.\n]{10,250}\bshall\b[^.\n]{5,250})',
            text_norm, re.MULTILINE,
        ):
            req = m.group(1).strip()
            if req not in frs:
                frs.append(req)

    # ── Non-Functional Requirements ───────────────────────────────────────────
    nfr_section_match = re.search(
        r'(?:^|\n)[ \t]*(?:\d[\d.]*\s+)?'
        r'(?:Non.?[Ff]unctional|Performance|Constraint|Quality|Safety|Security)'
        r'(?:\s+Requirements?)?\s*\n(.*?)'
        r'(?=\n[ \t]*(?:\d[\d.]*\s+)?[A-Z][a-zA-Z\s]{2,30}\n|\Z)',
        text_norm, re.DOTALL | re.IGNORECASE,
    )
    if nfr_section_match:
        for m in re.finditer(
            r'(?:^|\n)[ \t]*(?:\d+(?:\.\d+)+[ \t]+|[A-Z]{2,5}-\d+[: \t]+)(.{20,300})',
            nfr_section_match.group(1), re.MULTILINE,
        ):
            req = m.group(1).strip()
            if len(req) >= 20:
                nfrs.append(req)

    # ── Use Cases ─────────────────────────────────────────────────────────────
    uc_section_match = re.search(
        r'Use Cases?\s*\n(.*?)(?=\n[ \t]*(?:\d[\d.]*\s+)?[A-Z]|\Z)',
        text_norm, re.DOTALL | re.IGNORECASE,
    )
    if uc_section_match:
        for m in re.finditer(
            r'(?:Use\s*Case|UC|Scenario|Actor)[:\s-]*\d*[:\s]+(.{10,200})',
            uc_section_match.group(1), re.IGNORECASE,
        ):
            use_cases.append(m.group(1).strip())

    # Fallback: derive a use case from the Purpose/Scope section
    if not use_cases:
        scope_m = re.search(
            r'(?:Purpose|Scope|Overview|Introduction)[ \t]*\n([^\n]{30,500})',
            text_norm, re.IGNORECASE,
        )
        if scope_m:
            use_cases.append(scope_m.group(1).strip()[:200])

    return {
        'FRs': frs[:20],
        'NFRs': nfrs[:10],
        'useCases': use_cases[:10],
    }


def generate_specifier_golden(pure_doc):
    """Generate a specifier golden from a single PURE SRS document.

    Both the input (structured requirements extracted from the doc) and the
    expected output (the SRS document text) come from the same document,
    so the golden is internally consistent.
    """
    text = pure_doc['text']
    reqs = parse_pure_doc_requirements(text, pure_doc['name'])

    input_data = json.dumps({
        'collector_output': {
            'FRs': reqs['FRs'],
            'NFRs': reqs['NFRs'],
        },
        'analyzer_output': {
            'useCases': reqs['useCases'],
            'domainClasses': '',
            'businessRules': [],
            'dataModel': '',
            'traceability': [f'FR-{i + 1} -> UC-1' for i in range(len(reqs['FRs']))],
            'validation': [],
        },
    })

    return {
        'input': input_data,
        'actual_output': None,
        'expected_output': text[:5000],
        'context': [text[:1500]],
        'source_file': pure_doc['name'],
    }


# Generate for each split using PURE docs (not rSDE-Bench)
for split_name, docs in pure_splits.items():
    goldens = [generate_specifier_golden(doc) for doc in docs]
    out_path = GOLDENS_OUT / 'specifier_agent' / f'{split_name}.json'
    with open(out_path, 'w') as f:
        json.dump(goldens, f, indent=2)
    print(f'Specifier {split_name}: {len(goldens)} goldens -> {out_path}')

# Spot-check: verify input and output are about the same document
sample_doc = pure_splits['train'][0]
sg = generate_specifier_golden(sample_doc)
si = json.loads(sg['input'])
print(f'\nSpot-check: {sample_doc["name"]}')
print(f'  FRs extracted ({len(si["collector_output"]["FRs"])}):'
      f' {[fr[:60] for fr in si["collector_output"]["FRs"][:3]]}')
print(f'  NFRs extracted ({len(si["collector_output"]["NFRs"])}):'
      f' {[n[:60] for n in si["collector_output"]["NFRs"][:2]]}')
print(f'  Use cases ({len(si["analyzer_output"]["useCases"])}):'
      f' {[u[:60] for u in si["analyzer_output"]["useCases"][:2]]}')
print(f'  Expected output (first 200 chars): {sg["expected_output"][:200]}')

Specifier train: 18 goldens -> ../../data/goldens/specifier_agent/train.json
Specifier eval: 12 goldens -> ../../data/goldens/specifier_agent/eval.json

Spot-check: 2000 - nasa x38
  FRs extracted (20): ['Government Documents\t6', 'Non-Government Documents\t6', 'Required States and Modes\t7']
  NFRs extracted (0): []
  Use cases (0): []
  Expected output (first 200 chars): SOFTWARE REQUIREMENTS SPECIFICATION /
INTERFACE REQUIREMENTS SPECIFICATION

for the

X-38 Fault Tolerant System Services

Contract No. NAS 9-97216

DRL Sequence Nos. 12/14

12 April 2000

Prepared for


---
## 7. Generate Designer Agent Goldens
Input: SRS text. Expected output: `{systemArchitecture, fileStructure, componentDesign}`

Source: DevBench (has architecture_design.md + UML_class.md + UML_sequence.md)

In [22]:
def generate_designer_golden(project, requirements_text = None):
    """Generate a designer agent golden from a DevBench project."""
    # Input: Use the requirements text if provided, otherwise use the PRD as a proxy for SRS input
    input_text = project['PRD'] if requirements_text is None else requirements_text

    # Expected output: map DevBench artifacts to DesignerOutputModel
    arch = project.get('architecture_design', '')

    # Extract file structure from architecture_design if present
    file_structure = ''
    fs_match = re.search(r'```[\s\S]*?(\w+/[\s\S]*?)```', arch)
    if fs_match:
        file_structure = fs_match.group(1).strip()

    # Component design from UML class + sequence diagrams
    component_design = ''
    if project['UML_class']:
        component_design += project['UML_class']
    if project['UML_sequence']:
        component_design += '\n\n' + project['UML_sequence']

    expected_output = json.dumps({
        'systemArchitecture': arch,
        'fileStructure': file_structure if file_structure else 'See architecture design.',
        'componentDesign': component_design,
    })

    return {
        'input': input_text,
        'actual_output': None,
        'expected_output': expected_output,
        'context': [project['PRD']],
        'source_file': project['name'],
    }


for split_name, projects in devbench_splits.items():
    if split_name != 'benchmark':
        goldens = [generate_designer_golden(p) for p in projects]
        out_path = GOLDENS_OUT / 'designer_agent' / f'{split_name}.json'
        with open(out_path, 'w') as f:
            json.dump(goldens, f, indent=2)
        print(f'Designer {split_name}: {len(goldens)} goldens -> {out_path}')

Designer train: 4 goldens -> ../../data/goldens/designer_agent/train.json
Designer eval: 4 goldens -> ../../data/goldens/designer_agent/eval.json


---
## 8. Generate Documenter Agent Goldens
Input: DesignerOutputModel JSON. Expected output: Design document text.

In [23]:
def generate_documenter_golden(project):
    """Generate a documenter agent golden from a DevBench project."""
    # Input: DesignerOutputModel (reuse designer golden's expected_output)
    designer_golden = generate_designer_golden(project)
    input_data = designer_golden['expected_output']

    # Expected output: formatted design document
    # Use the concatenation of architecture_design.md, UML_class.md, and UML_sequence.md as the reference design document
    expected_output = project.get('architecture_design', '') + project.get('UML_class', '') + project.get('UML_sequence', '')
    if not expected_output:
        expected_output = f"# System Design Document\n\n## {project['name']}\n\nDesign document not available."

    return {
        'input': input_data,
        'actual_output': None,
        'expected_output': expected_output,
        'context': [],
        'source_file': project['name'],
    }


for split_name, projects in devbench_splits.items():
    if split_name != 'benchmark':
        goldens = [generate_documenter_golden(p) for p in projects]
        out_path = GOLDENS_OUT / 'documenter_agent' / f'{split_name}.json'
        with open(out_path, 'w') as f:
            json.dump(goldens, f, indent=2)
        print(f'Documenter {split_name}: {len(goldens)} goldens -> {out_path}')

Documenter train: 4 goldens -> ../../data/goldens/documenter_agent/train.json
Documenter eval: 4 goldens -> ../../data/goldens/documenter_agent/eval.json


---
## 9. Generate Multi and Single Agent Goldens
These are end-to-end goldens that test the full pipeline.

In [24]:
def extract_requirements_text(prd_text):
    """Extract requirements text from a DebBench PRD document."""
    input_text = 'Design a system with the following requirements: '
    
    # Regular expression that matches a heading line (##, ###, # …).
    heading_pat = re.compile(r'^(#{1,6})\s+(.*)$', re.MULTILINE)
    
    # A set to select only the required sections
    topic_sections = [
        'introduction', 'background', 'goals', 'features and functionalities',
        'constraints', 'technical constraints', 'requirements', 'dependencies',
    ]


    # Find all headings and their positions.
    matches = list(heading_pat.finditer(prd_text))

    sections: list[str] = []
    for i, match in enumerate(matches):
        heading = match.group(2).strip()
        if heading.lower() in topic_sections:
            start = match.end()
            end = matches[i + 1].start() if i + 1 < len(matches) else len(prd_text)
            section_body = prd_text[start:end].strip()
            if section_body:
                sections.append(section_body)

    input_text += " ".join(sections)
    return input_text

# Generate multi and single agent goldens
for split_name in ['train', 'eval', 'benchmark']:
    devb = devbench_splits[split_name]


    dw_goldens = []
    for i, proj in enumerate(devb):
        dw_goldens.append(generate_designer_golden(proj, extract_requirements_text(proj['PRD'])))

    with open(GOLDENS_OUT / 'read_agent' / f'{split_name}.json', 'w') as f:
        json.dump(dw_goldens, f, indent=2)
    print(f'READ Wrapper {split_name}: {len(dw_goldens)} goldens')

    # Single agent goldens
    with open(GOLDENS_OUT / 'single_agent' / f'{split_name}.json', 'w') as f:
        json.dump(dw_goldens, f, indent=2)
    print(f'Single {split_name}: {len(dw_goldens)} goldens')

READ Wrapper train: 4 goldens
Single train: 4 goldens
READ Wrapper eval: 4 goldens
Single eval: 4 goldens
READ Wrapper benchmark: 14 goldens
Single benchmark: 14 goldens


---
## 10. Summary: Golden Counts

In [25]:
# Summarize all generated goldens
summary = []
for agent_dir in sorted(GOLDENS_OUT.iterdir()):
    if agent_dir.is_dir():
        for split_file in sorted(agent_dir.glob('*.json')):
            # Skip legacy golden files
            if split_file.stem not in ('train', 'eval', 'benchmark'):
                continue
            with open(split_file) as f:
                goldens = json.load(f)
            summary.append({
                'Agent': agent_dir.name,
                'Split': split_file.stem,
                'Count': len(goldens),
            })

summary_df = pd.DataFrame(summary)
pivot = summary_df.pivot(index='Agent', columns='Split', values='Count').fillna(0).astype(int)
pivot = pivot[['train', 'eval', 'benchmark']]  # Reorder columns
pivot['Total'] = pivot.sum(axis=1)
print(pivot.to_string())
print(f'\nTotal goldens: {pivot["Total"].sum()}')

Split             train  eval  benchmark  Total
Agent                                          
analyzer_agent       33    20          0     53
collector_agent      33    20          0     53
designer_agent        4     4          0      8
documenter_agent      4     4          0      8
read_agent            4     4         14     22
single_agent          4     4         14     22
specifier_agent      18    12          0     30

Total goldens: 196
